## 1. Ontario Open Data – “Wages by Education Level”
### Dataset Columns:

- <b>_id</b>: Unique identification number for each record.

- <b>YEAR</b>: The year in which the data was recorded (ranging from 2006 to 2024).
- <b>GEOGRAPHY</b>: The geographic region represented in the data (e.g., Canada, Ontario).
- <b>IMMIGRANT</b>: Immigration status category.
    - `Total` — entire population (all immigration groups combined).
    - `Born in Canada` — people born in Canada.
    - `Total Landed Immigrants` — all permanent residents (landed).
    - `Very recent immigrants (<=5 yrs)`
    - `Recent immigrants (5–10 yrs)`
    - `Established immigrants (10+ yrs)`
    - `Non-landed immigrants` — temporary residents (students, temporary workers).

- <b>TYPE OF WORK</b>: Employment type.
    - `All employees` — all employed persons (both full-time and part-time).
    - `Full-time` — typically defined as ~30+ hours/week (check source metadata; definition may vary).
    - `Part-time` — fewer hours than full-time.
- <b>WAGE RATE</b>: Type of wage measure used — in this dataset, it represents the Median hourly wage.
- <b>EDUCATION</b>: Educational attainment category.
    - `Total, all education levels` (aggregate)
    - `0–8 years` - People who did not complete high school, just middle or elemenatry school.
    - `Some high school`- Went to high school but did not graduate from high school.
    - `High school graduate`- Completed high school
    - `Some post-secondary` - Completed some form of postsecondary education either college or partial university.
    - `Post-secondary certificate or diploma` - Graduated from a non- university program
    - `Bachelor's degree` - Completed a university undergraduate program.
    - `Above bachelor's degree` - This could include master's, professional, and doctoral degrees.
- <b>AGE GROUP</b>: Age range category.
    - `15+` — age 15 and older (broad working-age start).
    - `25+` — age 25 and older (prime-age cutoff).
    - `25–34` — young adult workers.
    - `25–54` — core working-age group.
    - `25–64` — extended working-age (excludes 65+).
- <b>Both sexes</b>: Reported wage rate for the total population (both male and female combined).
- <b>Men</b>: Reported wage rate for male employees.
- <b>Women</b>: Reported wage rate for female employees.

## **1.1.** Data Preprocessing and Cleaning:

#### Importing all the necessary Libraries needed.

In [2]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

**1.1.1.** Reading in the Data:

In [3]:
education_level = pd.read_csv('../raw_data/education_level_raw.csv')
education_level.head()

,_id,YEAR,GEOGRAPHY,IMMIGRANT,TYPE OF WORK,WAGE RATE,EDUCATION,AGE GROUP,Both sexes,Men,Women
0,1,2006,Canada,Total,All employees,Median hourly wage,"Total, all education levels",15 +,17.5,19.2,16.0
1,2,2006,Canada,Total,All employees,Median hourly wage,"Total, all education levels",25 +,19.4,21.5,17.5
2,3,2006,Canada,Total,All employees,Median hourly wage,"Total, all education levels",25 - 34,18.0,19.0,16.8
3,4,2006,Canada,Total,All employees,Median hourly wage,"Total, all education levels",25 - 54,19.5,21.5,17.8
4,5,2006,Canada,Total,All employees,Median hourly wage,"Total, all education levels",25 - 64,19.5,21.5,17.6


**1.1.2.** Converting all the column names to lowercase to ensure uniformity.

In [3]:
education_level.columns = education_level.columns.str.lower()

**1.1.3.** Checking for detailed information about the data

In [4]:
education_level.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41040 entries, 0 to 41039
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   _id           41040 non-null  int64  
 1   year          41040 non-null  int64  
 2   geography     41040 non-null  object 
 3   immigrant     41040 non-null  object 
 4   type of work  41040 non-null  object 
 5   wage rate     41040 non-null  object 
 6   education     41040 non-null  object 
 7   age group     41040 non-null  object 
 8   both sexes    41040 non-null  float64
 9   men           41040 non-null  float64
 10  women         41040 non-null  float64
dtypes: float64(3), int64(2), object(6)
memory usage: 3.4+ MB


**1.1.4.** Check for missing values:

In [5]:
education_level_missing = education_level.isnull().sum()
education_level_missing

,0
_id,0
year,0
geography,0
immigrant,0
type of work,0
wage rate,0
education,0
age group,0
both sexes,0
men,0


**1.1.5.** Categorizing the 'education' column.

In [6]:
def categorize_education(education):
    education = education.strip()

    if education in ['0 - 8 years', 'Some high school', 'Less than High School']:
        return 'Less than high school'
    elif education in ['High school graduate']:
        return 'High school diploma'
    elif education in ['Some post-secondary', 'Post-secondary certificate or diploma', 'College or Trade Diploma']:
        return 'Postsecondary certificate or diploma'
    elif education in ["Bachelor's degree", "Above bachelor’s degree", "University degree"]:
        return 'University degree'
    elif education in ['Total, all education levels']:
        return None
    else:
        return None

education_level['education'] = education_level['education'].apply(categorize_education)


**1.1.6** Removing the 'None' option from the 'education' column.

In [7]:
education_level = education_level.dropna(subset=['education'])
education_level['education'].unique()

array(['Less than high school', 'High school diploma',
       'Postsecondary certificate or diploma', 'University degree'],
      dtype=object)

**1.1.7** Filtering the 'geography' column to pick just Ontario.

In [8]:
education_level = education_level[education_level['geography'].str.strip() == 'Ontario'].reset_index(drop=True)
education_level['geography'].unique()

array([' Ontario'], dtype=object)

**1.1.8.** Standardizing the 'immigration' column in the dataset.

**1.1.8.1.** Removing the extra spaces.

In [9]:
education_level['immigrant'] = education_level['immigrant'].str.strip()

**1.1.8.2.** Dropping the aggregate totals to avoid double counting.

In [10]:
education_level = education_level[
    ~education_level['immigrant'].isin(['Total', 'Total Landed Immigrants'])
].reset_index(drop=True)
education_level['immigrant'].unique()

array(['Very recent immigrants, 5 years or less',
       'Recent immigrants 5+ years', 'Recent immigrants, 5+ to 10 years',
       'Established immigrants, 10+ years', 'Non-landed immigrants',
       'Born in Canada'], dtype=object)

**1.1.8.3.** Mapping the categories into two consistent groups.

In [11]:
education_level['immigrant'] = education_level['immigrant'].replace({
    'Born in Canada': 'Born in Canada (non-immigrant)',
    'Very recent immigrants, 5 years or less': 'Immigrant',
    'Recent immigrants 5+ years': 'Immigrant',
    'Recent immigrants, 5+ to 10 years': 'Immigrant',
    'Established immigrants, 10+ years': 'Immigrant',
    'Non-landed immigrants': 'Immigrant'
})

In [12]:
education_level['immigrant'].unique()

array(['Immigrant', 'Born in Canada (non-immigrant)'], dtype=object)

**1.1.9.** Checking the final outcome of the first 5 rows after cleaning the "Wages by Education" Data.

In [13]:
education_level.head()

,_id,year,geography,immigrant,type of work,wage rate,education,age group,both sexes,men,women
0,1356,2006,Ontario,Immigrant,All employees,Median hourly wage,Less than high school,15 +,11.0,11.9,0.0
1,1357,2006,Ontario,Immigrant,All employees,Median hourly wage,Less than high school,25 +,12.0,13.3,0.0
2,1358,2006,Ontario,Immigrant,All employees,Median hourly wage,Less than high school,25 - 34,0.0,0.0,0.0
3,1359,2006,Ontario,Immigrant,All employees,Median hourly wage,Less than high school,25 - 54,12.5,15.0,0.0
4,1360,2006,Ontario,Immigrant,All employees,Median hourly wage,Less than high school,25 - 64,12.5,13.3,0.0


**1.1.10.** Checking descriptive statistics of the data:

In [14]:
education_level.describe()

,_id,year,both sexes,men,women
count,11970.000000,11970.000000,11970.000000,11970.000000,11970.000000
mean,21195.500000,2015.000000,16.593308,15.612757,13.577552
std,11833.609214,5.477454,9.482660,12.104068,10.072793
min,1356.000000,2006.000000,0.000000,0.000000,0.000000
25%,10598.250000,2010.000000,12.500000,0.000000,0.000000
50%,21195.500000,2015.000000,17.000000,18.000000,15.000000
75%,31792.750000,2020.000000,22.000000,24.000000,20.000000
max,41035.000000,2024.000000,46.700000,50.000000,44.200000


**1.1.11.** Checking for the number of unique values in the dataset.

In [15]:
education_level.nunique()

,0
_id,11970
year,19
geography,1
immigrant,2
type of work,3
wage rate,1
education,4
age group,5
both sexes,331
men,358
